<a href="https://colab.research.google.com/github/maheshvlfm-coder/Forecasting-/blob/main/Forecasting_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Demand vs Sales Analysis & Forecasting Tool**

1.Data Upload: Users upload historical demand and sales data via a CSV file interface. A sample file is available for guidance.

2.Pattern Analysis: The tool analyzes demand and sales time series for patterns, calculating key metrics and identifying trend and seasonality.

3.Visualization & Reporting: Analysis results are visualized through plots and compiled into a downloadable Excel report.

4.Model Suggestion: Forecasting models are suggested based on detected sales patterns.

5.Forecast Generation: Users select a model and horizon to generate future sales forecasts, including model validation and visualization.

6.Comprehensive Report: A downloadable report containing forecast summaries and full historical/forecasted data is created.

In [ ]:
# 📈 Demand vs Sales Analysis & Forecasting Tool - Google Colab
# Tool for analyzing demand vs actual sales patterns and generating forecasts

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import io
import base64
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")

print("📈 Installing required packages...")
# Install required packages
!pip install statsmodels prophet scikit-learn ipywidgets plotly -q

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from google.colab import files

# Import forecasting libraries
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("✅ All packages installed successfully!")
print("=" * 70)
print("🚀 DEMAND vs SALES ANALYSIS & FORECASTING TOOL")
print("=" * 70)

class DemandSalesAnalysisTool:
    def __init__(self):
        self.data = None
        self.analysis_results = {}
        self.forecast_results = {}
        self.selected_sku = None

    def create_sample_file(self):
        """Create sample file for download"""
        sample_data = """SKU_1,,
Period,Demand,Sales
1,18,66
2,54,50
3,79,31
4,41,93
5,85,46
6,55,92
7,30,77
8,50,35
9,65,36
10,77,31
11,32,22
12,75,76
13,56,94
14,93,17
15,58,60
16,45,72
17,92,71
18,69,70
19,46,18
20,87,88
21,35,43
22,49,73
23,90,28
24,35,23
25,76,53
26,45,64
27,39,95
28,12,46
29,83,94
30,38,63
31,29,87
32,24,82
33,36,76
34,34,25
35,52,53
36,84,94"""

        # Create download link
        b64 = base64.b64encode(sample_data.encode()).decode()
        filename = "sample_demand_sales_data.csv"
        download_link = f'<a download="{filename}" href="data:text/csv;base64,{b64}">📥 Download Sample File</a>'

        print("📄 Sample File Format:")
        print("- SKU name in first row, first column")
        print("- Columns: Period, Demand, Sales")
        print("- Period: Sequential numbers (1, 2, 3...)")
        print("- At least 24 periods recommended for good analysis")
        print()
        display(HTML(download_link))

    def start_analysis(self):
        """Start the analysis workflow"""
        print("📋 STEP 1: Upload Data File")
        print("=" * 40)
        print("Expected CSV format: SKU name in header, then Period, Demand, Sales columns")
        print("-" * 40)

        # Create sample file download button
        sample_button = widgets.Button(
            description='📄 Download Sample File',
            button_style='info',
            layout=widgets.Layout(width='200px', height='40px')
        )

        def download_sample(b):
            clear_output(wait=True)
            print("📄 SAMPLE FILE DOWNLOAD")
            print("=" * 30)
            self.create_sample_file()
            print("\n" + "="*50)
            self.start_analysis()

        sample_button.on_click(download_sample)

        # Create file upload interface
        upload_widget = widgets.FileUpload(
            accept='.csv',
            multiple=False,
            description='📁 Upload CSV File'
        )

        process_button = widgets.Button(
            description='📊 Process & Analyze',
            button_style='success',
            layout=widgets.Layout(width='200px', height='40px')
        )

        def process_file(b):
            if upload_widget.value:
                try:
                    # Get uploaded file
                    filename = list(upload_widget.value.keys())[0]
                    content = upload_widget.value[filename]['content']

                    # Save and read file
                    with open('uploaded_data.csv', 'wb') as f:
                        f.write(content)

                    # Read file with special format handling
                    self.data = self.read_special_format('uploaded_data.csv')

                    if self.data is None:
                        print("❌ Error reading file. Please check format.")
                        return

                    print("✅ File uploaded successfully!")
                    print(f"📊 Data shape: {self.data.shape}")
                    print(f"📦 SKU found: {self.data['SKU'].iloc[0]}")
                    print(f"📅 Periods: {self.data['Period'].min()} to {self.data['Period'].max()}")

                    clear_output(wait=True)
                    self.analyze_demand_vs_sales()

                except Exception as e:
                    print(f"❌ Error processing file: {str(e)}")
            else:
                print("❌ Please select a file first!")

        process_button.on_click(process_file)

        display(widgets.HBox([sample_button, upload_widget]))
        display(process_button)

    def read_special_format(self, filepath):
        """Read the special format CSV file"""
        try:
            # Read the file line by line
            with open(filepath, 'r') as f:
                lines = f.readlines()

            # Extract SKU name from first line
            sku_name = lines[0].split(',')[0].strip()

            # Read the data part (skip first line)
            data_lines = lines[1:]

            # Create DataFrame
            data_rows = []
            for line in data_lines:
                parts = line.strip().split(',')
                if len(parts) >= 3 and parts[0].strip() != 'Period':  # Skip header
                    try:
                        period = int(parts[0].strip())
                        demand = float(parts[1].strip())
                        sales = float(parts[2].strip())
                        data_rows.append({
                            'SKU': sku_name,
                            'Period': period,
                            'Demand': demand,
                            'Sales': sales
                        })
                    except (ValueError, IndexError):
                        continue

            if data_rows:
                df = pd.DataFrame(data_rows)
                return df
            else:
                return None

        except Exception as e:
            print(f"Error reading file: {str(e)}")
            return None

    def analyze_demand_vs_sales(self):
        """STEP 2: Analyze demand vs actual sales patterns"""
        print("📊 STEP 2: DEMAND vs SALES PATTERN ANALYSIS")
        print("=" * 50)

        if self.data is None:
            print("❌ No data available for analysis")
            return

        # Get SKU name
        sku = self.data['SKU'].iloc[0]
        print(f"🔍 Analyzing {sku}...")

        # Calculate metrics
        sku_analysis = self.calculate_sku_metrics(sku, self.data)

        # Store detailed analysis
        self.analysis_results[sku] = sku_analysis

        # Create comprehensive analysis report
        self.create_analysis_report(sku_analysis)

        # Display visualizations
        self.create_analysis_visualizations()

        # Provide download button for analysis
        self.create_analysis_download_button()

        # Move to forecasting step
        self.suggest_forecasting_models()

    def calculate_sku_metrics(self, sku, sku_data):
        """Calculate level, trend, seasonality, MAPE, RMSE for demand vs sales"""
        try:
            # Prepare time series data
            demand_ts = pd.Series(sku_data['Demand'].values, index=sku_data['Period'])
            sales_ts = pd.Series(sku_data['Sales'].values, index=sku_data['Period'])

            # Calculate basic metrics
            demand_mean = demand_ts.mean()
            sales_mean = sales_ts.mean()
            fulfillment_rate = (sales_ts.sum() / demand_ts.sum()) * 100 if demand_ts.sum() > 0 else 0

            # Calculate MAPE and RMSE (Sales vs Demand)
            mask = demand_ts > 0  # Avoid division by zero
            valid_demand = demand_ts[mask]
            valid_sales = sales_ts[mask]

            if len(valid_demand) > 0:
                mape = np.mean(np.abs((valid_demand - valid_sales) / valid_demand)) * 100
                rmse = np.sqrt(mean_squared_error(valid_demand, valid_sales))
                mae = mean_absolute_error(valid_demand, valid_sales)
            else:
                mape = rmse = mae = np.nan

            # Demand Analysis - Decomposition
            demand_analysis = self.analyze_time_series_components(demand_ts, "Demand")

            # Sales Analysis - Decomposition
            sales_analysis = self.analyze_time_series_components(sales_ts, "Sales")

            return {
                'SKU': sku,
                'Data_Points': len(sku_data),
                'Period_Range': f"Period {sku_data['Period'].min()} to {sku_data['Period'].max()}",
                'Demand_Mean': round(demand_mean, 2),
                'Sales_Mean': round(sales_mean, 2),
                'Fulfillment_Rate_%': round(fulfillment_rate, 2),
                'MAPE_%': round(mape, 2) if not np.isnan(mape) else 'N/A',
                'RMSE': round(rmse, 2) if not np.isnan(rmse) else 'N/A',
                'MAE': round(mae, 2) if not np.isnan(mae) else 'N/A',
                'Demand_Analysis': demand_analysis,
                'Sales_Analysis': sales_analysis,
                'Raw_Data': sku_data
            }

        except Exception as e:
            print(f"❌ Error analyzing {sku}: {str(e)}")
            return {'SKU': sku, 'Error': str(e)}

    def analyze_time_series_components(self, ts_data, data_type):
        """Analyze level, trend, and seasonality of time series"""
        try:
            # Level (overall mean)
            level = ts_data.mean()

            # Simple trend using linear regression
            x = np.arange(len(ts_data))
            trend_slope = np.polyfit(x, ts_data.values, 1)[0]

            if abs(trend_slope) < 0.1:
                trend_type = "No Significant Trend"
            elif trend_slope > 0:
                trend_type = f"Positive Trend (+{trend_slope:.3f} per period)"
            else:
                trend_type = f"Negative Trend ({trend_slope:.3f} per period)"

            # Seasonality analysis (if enough data)
            seasonality_strength = 'N/A'
            seasonality_type = 'Insufficient data for seasonality analysis'

            if len(ts_data) >= 24:  # Need at least 2 cycles
                try:
                    # Try to detect seasonality with period=12
                    decomposition = seasonal_decompose(ts_data, model='additive', period=12)
                    seasonality_strength = decomposition.seasonal.std() / ts_data.std()

                    if seasonality_strength > 0.3:
                        seasonality_type = f"Strong Seasonality (strength: {seasonality_strength:.3f})"
                    elif seasonality_strength > 0.1:
                        seasonality_type = f"Moderate Seasonality (strength: {seasonality_strength:.3f})"
                    else:
                        seasonality_type = f"Weak Seasonality (strength: {seasonality_strength:.3f})"

                except:
                    # If seasonal decomposition fails, use simple seasonal check
                    seasonality_strength = 0.1  # Default moderate
                    seasonality_type = "Basic seasonality assumed"

            # Variability
            cv = ts_data.std() / ts_data.mean() if ts_data.mean() > 0 else 0

            return {
                'Level': round(level, 2),
                'Trend_Slope': round(trend_slope, 4),
                'Trend_Type': trend_type,
                'Seasonality_Strength': seasonality_strength if seasonality_strength != 'N/A' else 0.1,
                'Seasonality_Type': seasonality_type,
                'Variability_CV': round(cv, 3),
                'Has_Trend': abs(trend_slope) > 0.1,
                'Has_Seasonality': seasonality_strength != 'N/A' and seasonality_strength > 0.1
            }

        except Exception as e:
            return {
                'Level': round(ts_data.mean(), 2),
                'Trend_Type': 'Analysis Failed',
                'Seasonality_Type': 'Analysis Failed',
                'Error': str(e)
            }

    def create_analysis_report(self, analysis):
        """Create comprehensive analysis report"""
        print("\n📋 ANALYSIS SUMMARY REPORT")
        print("=" * 50)

        if 'Error' not in analysis:
            print(f"🔍 SKU: {analysis['SKU']}")
            print(f"📊 Data Points: {analysis['Data_Points']}")
            print(f"📅 Period Range: {analysis['Period_Range']}")
            print(f"📈 Average Demand: {analysis['Demand_Mean']}")
            print(f"📦 Average Sales: {analysis['Sales_Mean']}")
            print(f"✅ Fulfillment Rate: {analysis['Fulfillment_Rate_%']}%")
            print(f"📏 MAPE (Sales vs Demand): {analysis['MAPE_%']}%")
            print(f"📐 RMSE: {analysis['RMSE']}")
            print(f"📊 MAE: {analysis['MAE']}")

            demand_analysis = analysis['Demand_Analysis']
            sales_analysis = analysis['Sales_Analysis']

            print(f"\n📈 DEMAND PATTERN ANALYSIS:")
            print(f"  - Level: {demand_analysis['Level']}")
            print(f"  - Trend: {demand_analysis['Trend_Type']}")
            print(f"  - Seasonality: {demand_analysis['Seasonality_Type']}")
            print(f"  - Variability (CV): {demand_analysis['Variability_CV']}")

            print(f"\n📦 SALES PATTERN ANALYSIS:")
            print(f"  - Level: {sales_analysis['Level']}")
            print(f"  - Trend: {sales_analysis['Trend_Type']}")
            print(f"  - Seasonality: {sales_analysis['Seasonality_Type']}")
            print(f"  - Variability (CV): {sales_analysis['Variability_CV']}")

            # Interpretation
            print(f"\n💡 INTERPRETATION:")
            if analysis['MAPE_%'] != 'N/A':
                if analysis['MAPE_%'] < 20:
                    print("  ✅ Good alignment between demand and sales")
                elif analysis['MAPE_%'] < 40:
                    print("  ⚠️  Moderate alignment between demand and sales")
                else:
                    print("  ❌ Poor alignment between demand and sales")

            if analysis['Fulfillment_Rate_%'] > 90:
                print("  ✅ Excellent demand fulfillment")
            elif analysis['Fulfillment_Rate_%'] > 70:
                print("  ⚠️  Good demand fulfillment")
            else:
                print("  ❌ Poor demand fulfillment")

    def create_analysis_visualizations(self):
        """Create comprehensive visualizations"""
        print("\n📊 CREATING VISUALIZATIONS...")

        sku = self.data['SKU'].iloc[0]
        periods = self.data['Period'].values
        demand = self.data['Demand'].values
        sales = self.data['Sales'].values

        # Create comprehensive dashboard
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle(f'Demand vs Sales Analysis - {sku}', fontsize=16, fontweight='bold')

        # 1. Time series plot
        ax1 = axes[0, 0]
        ax1.plot(periods, demand, label='Demand', linewidth=2, marker='o', color='blue', markersize=4)
        ax1.plot(periods, sales, label='Sales', linewidth=2, marker='s', color='red', markersize=4, alpha=0.8)
        ax1.set_title('Demand vs Sales Over Time', fontweight='bold')
        ax1.set_xlabel('Period')
        ax1.set_ylabel('Quantity')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        # 2. Scatter plot - Demand vs Sales correlation
        ax2 = axes[0, 1]
        ax2.scatter(demand, sales, alpha=0.7, color='purple', s=50)

        # Add perfect correlation line
        max_val = max(demand.max(), sales.max())
        min_val = min(demand.min(), sales.min())
        ax2.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, label='Perfect Match')

        # Add trend line
        z = np.polyfit(demand, sales, 1)
        p = np.poly1d(z)
        ax2.plot(demand, p(demand), "r--", alpha=0.8, label=f'Trend Line')

        ax2.set_title('Demand vs Sales Correlation', fontweight='bold')
        ax2.set_xlabel('Demand')
        ax2.set_ylabel('Sales')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        # 3. Residual analysis (Sales - Demand)
        ax3 = axes[1, 0]
        residuals = sales - demand
        ax3.plot(periods, residuals, color='green', marker='o', linewidth=2, markersize=4)
        ax3.axhline(y=0, color='black', linestyle='--', alpha=0.5)
        ax3.set_title('Residuals (Sales - Demand)', fontweight='bold')
        ax3.set_xlabel('Period')
        ax3.set_ylabel('Residuals')
        ax3.grid(True, alpha=0.3)

        # 4. Distribution comparison
        ax4 = axes[1, 1]
        ax4.hist(demand, bins=15, alpha=0.7, label='Demand', color='blue', density=True)
        ax4.hist(sales, bins=15, alpha=0.7, label='Sales', color='red', density=True)
        ax4.set_title('Distribution Comparison', fontweight='bold')
        ax4.set_xlabel('Quantity')
        ax4.set_ylabel('Density')
        ax4.legend()
        ax4.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

    def create_analysis_download_button(self):
        """Create download button for analysis results"""
        print("\n💾 DOWNLOAD ANALYSIS RESULTS")
        print("=" * 40)

        download_button = widgets.Button(
            description='📥 Download Analysis Report',
            button_style='info',
            layout=widgets.Layout(width='250px', height='40px')
        )

        def download_analysis(b):
            try:
                # Create comprehensive Excel file
                output = io.BytesIO()

                sku = list(self.analysis_results.keys())[0]
                analysis = self.analysis_results[sku]

                with pd.ExcelWriter(output, engine='openpyxl') as writer:
                    # Summary sheet
                    summary_data = {
                        'Metric': ['SKU', 'Data Points', 'Period Range', 'Average Demand', 'Average Sales',
                                  'Fulfillment Rate (%)', 'MAPE (%)', 'RMSE', 'MAE',
                                  'Demand Level', 'Demand Trend', 'Demand Seasonality', 'Demand Variability',
                                  'Sales Level', 'Sales Trend', 'Sales Seasonality', 'Sales Variability'],
                        'Value': [
                            analysis['SKU'],
                            analysis['Data_Points'],
                            analysis['Period_Range'],
                            analysis['Demand_Mean'],
                            analysis['Sales_Mean'],
                            analysis['Fulfillment_Rate_%'],
                            analysis['MAPE_%'],
                            analysis['RMSE'],
                            analysis['MAE'],
                            analysis['Demand_Analysis']['Level'],
                            analysis['Demand_Analysis']['Trend_Type'],
                            analysis['Demand_Analysis']['Seasonality_Type'],
                            analysis['Demand_Analysis']['Variability_CV'],
                            analysis['Sales_Analysis']['Level'],
                            analysis['Sales_Analysis']['Trend_Type'],
                            analysis['Sales_Analysis']['Seasonality_Type'],
                            analysis['Sales_Analysis']['Variability_CV']
                        ]
                    }

                    summary_df = pd.DataFrame(summary_data)
                    summary_df.to_excel(writer, sheet_name='Analysis_Summary', index=False)

                    # Raw data with additional calculations
                    raw_data = analysis['Raw_Data'].copy()
                    raw_data['Residual_Sales_Demand'] = raw_data['Sales'] - raw_data['Demand']
                    raw_data['Absolute_Error'] = abs(raw_data['Residual_Sales_Demand'])
                    raw_data['Percentage_Error'] = abs((raw_data['Sales'] - raw_data['Demand']) / raw_data['Demand']) * 100
                    raw_data.to_excel(writer, sheet_name='Raw_Data_Analysis', index=False)

                output.seek(0)

                # Create download link
                b64 = base64.b64encode(output.getvalue()).decode()
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                filename = f"demand_sales_analysis_{sku}_{timestamp}.xlsx"

                download_link = f'<a download="{filename}" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64}">Click here to download analysis report</a>'
                display(HTML(download_link))

                print("✅ Analysis report ready for download!")

            except Exception as e:
                print(f"❌ Error creating download file: {str(e)}")

        download_button.on_click(download_analysis)
        display(download_button)

    def suggest_forecasting_models(self):
        """STEP 3: Suggest forecasting models based on sales patterns"""
        print("\n\n🤖 STEP 3: FORECASTING MODEL SUGGESTIONS")
        print("=" * 60)
        print("Based on sales pattern analysis, here are the recommended models:")
        print("-" * 60)

        sku = list(self.analysis_results.keys())[0]
        analysis = self.analysis_results[sku]
        sales_analysis = analysis['Sales_Analysis']

        suggestions = self.get_model_suggestions_for_sku(sku, sales_analysis)

        print(f"\n🎯 Recommendations for {sku}:")
        for i, suggestion in enumerate(suggestions, 1):
            emoji = "🥇" if i == 1 else "🥈" if i == 2 else "🥉"
            print(f"  {emoji} {suggestion['model']} - {suggestion['reason']}")

        self.model_suggestions = {sku: suggestions}
        self.create_forecasting_interface()

    def get_model_suggestions_for_sku(self, sku, sales_analysis):
        """Get model suggestions for specific SKU based on sales pattern"""
        suggestions = []

        # Check characteristics
        has_trend = sales_analysis.get('Has_Trend', False)
        has_seasonality = sales_analysis.get('Has_Seasonality', False)
        variability = sales_analysis.get('Variability_CV', 0)

        # Model suggestions based on pattern
        if not has_trend and not has_seasonality:
            suggestions.extend([
                {'model': 'Simple Exponential Smoothing', 'reason': 'Stable pattern - no trend or seasonality'},
                {'model': 'Moving Average', 'reason': 'Good for stable demand'},
                {'model': 'Naive', 'reason': 'Simple baseline model'}
            ])

        elif has_trend and not has_seasonality:
            suggestions.extend([
                {'model': 'Double Exponential Smoothing', 'reason': 'Handles trend effectively'},
                {'model': 'Linear Trend', 'reason': 'Clear trend pattern detected'},
                {'model': 'ARIMA', 'reason': 'Flexible for trending data'}
            ])

        elif has_seasonality:
            suggestions.extend([
                {'model': 'Triple Exponential Smoothing', 'reason': 'Handles seasonality and trend'},
                {'model': 'Prophet', 'reason': 'Excellent for seasonal patterns'},
                {'model': 'Seasonal ARIMA', 'reason': 'Statistical approach for seasonality'}
            ])

        # Always add versatile models
        if 'ARIMA' not in [s['model'] for s in suggestions]:
            suggestions.append({'model': 'ARIMA', 'reason': 'Versatile statistical model'})

        if 'Prophet' not in [s['model'] for s in suggestions]:
            suggestions.append({'model': 'Prophet', 'reason': 'Robust modern forecasting'})

        return suggestions[:4]  # Return top 4 suggestions

    def create_forecasting_interface(self):
        """STEP 4: Create forecasting model selection interface"""
        print("\n\n🔧 STEP 4: SELECT FORECASTING MODEL & GENERATE FORECAST")
        print("=" * 70)

        sku = list(self.analysis_results.keys())[0]

        # Model selection
        model_options = [s['model'] for s in self.model_suggestions[sku]]
        model_dropdown = widgets.Dropdown(
            options=model_options,
            value=model_options[0],
            description='Forecasting Model:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='350px')
        )

        # Forecast horizon
        horizon_slider = widgets.IntSlider(
            value=6,
            min=1,
            max=24,
            step=1,
            description='Forecast Periods:',
            style={'description_width': 'initial'}
        )

        # Generate forecast button
        forecast_button = widgets.Button(
            description='🚀 Generate Forecast',
            button_style='success',
            layout=widgets.Layout(width='200px', height='40px')
        )

        # Results container
        results_output = widgets.Output()

        def generate_forecast(b):
            with results_output:
                clear_output(wait=True)
                try:
                    selected_model = model_dropdown.value
                    forecast_horizon = horizon_slider.value

                    print(f"🚀 Generating forecast for {sku} using {selected_model}")
                    print("=" * 60)

                    # Run forecasting
                    forecast_result = self.run_forecast_model(sku, selected_model, forecast_horizon)

                    if forecast_result:
                        # Display results
                        self.display_forecast_results(forecast_result)

                        # Create download button for forecast
                        self.create_forecast_download_button(forecast_result)
                    else:
                        print("❌ Forecasting failed. Please try different parameters.")

                except Exception as e:
                    print(f"❌ Error generating forecast: {str(e)}")

        forecast_button.on_click(generate_forecast)

        # Display interface
        interface = widgets.VBox([
            widgets.HTML("<h3>🎯 Configure Forecasting Parameters</h3>"),
            model_dropdown,
            horizon_slider,
            widgets.HTML("<br>"),
            forecast_button,
            widgets.HTML("<br>"),
            results_output
        ])

        display(interface)

    def run_forecast_model(self, sku, model_name, forecast_horizon):
        """Run the selected forecasting model"""
        try:
            # Get SKU data
            sku_data = self.data

            # Use sales data for forecasting (actual historical performance)
            sales_values = sku_data['Sales'].values
            periods = sku_data['Period'].values

            # Create time series
            sales_ts = pd.Series(sales_values, index=periods)

            # Split for validation (last 6 periods)
            split_point = max(1, len(sales_ts) - 6)
            train_sales = sales_ts[:split_point]
            test_sales = sales_ts[split_point:] if split_point < len(sales_ts) else pd.Series()

            # Run forecasting model
            if model_name == 'Simple Exponential Smoothing':
                model = ExponentialSmoothing(train_sales, trend=None, seasonal=None)
                fitted_model = model.fit(smoothing_level=0.3)

            elif model_name == 'Double Exponential Smoothing':
                model = ExponentialSmoothing(train_sales, trend='add', seasonal=None)
                fitted_model = model.fit(smoothing_level=0.3, smoothing_trend=0.3)

            elif model_name == 'Triple Exponential Smoothing':
                try:
                    if len(train_sales) >= 24:  # Need enough data for seasonality
                        model = ExponentialSmoothing(train_sales, trend='add', seasonal='add', seasonal_periods=12)
                        fitted_model = model.fit(smoothing_level=0.3, smoothing_trend=0.3, smoothing_seasonal=0.3)
                    else:
                        # Fallback to double smoothing if not enough data
                        model = ExponentialSmoothing(train_sales, trend='add', seasonal=None)
                        fitted_model = model.fit(smoothing_level=0.3, smoothing_trend=0.3)
                except:
                    model = ExponentialSmoothing(train_sales, trend='add', seasonal=None)
                    fitted_model = model.fit(smoothing_level=0.3, smoothing_trend=0.3)

            elif model_name == 'ARIMA':
                model = ARIMA(train_sales, order=(1, 1, 1))
                fitted_model = model.fit()

            elif model_name == 'Seasonal ARIMA':
                try:
                    model = ARIMA(train_sales, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))
                    fitted_model = model.fit()
                except:
                    # Fallback to regular ARIMA
                    model = ARIMA(train_sales, order=(1, 1, 1))
                    fitted_model = model.fit()

            elif model_name == 'Prophet':
                # Create date-like index for Prophet
                start_date = pd.Timestamp('2020-01-01')
                dates = [start_date + pd.DateOffset(months=i) for i in range(len(train_sales))]

                prophet_data = pd.DataFrame({
                    'ds': dates,
                    'y': train_sales.values
                })
                model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
                fitted_model = model.fit(prophet_data)

            elif model_name == 'Moving Average':
                window = min(3, len(train_sales) // 2)
                forecast_values = [train_sales.tail(window).mean()] * forecast_horizon
                train_forecast = train_sales.rolling(window=window).mean()
                test_forecast = [train_sales.tail(window).mean()] * len(test_sales) if len(test_sales) > 0 else []

                return self.compile_forecast_results(sku, model_name, sales_ts, train_sales,
                                                   test_sales, train_forecast, test_forecast,
                                                   forecast_values, forecast_horizon)

            elif model_name == 'Linear Trend':
                x = np.arange(len(train_sales))
                coeffs = np.polyfit(x, train_sales.values, 1)

                # Generate forecasts
                train_forecast = pd.Series(np.polyval(coeffs, x), index=train_sales.index)

                x_future = np.arange(len(train_sales), len(train_sales) + forecast_horizon)
                forecast_values = np.polyval(coeffs, x_future).tolist()

                if len(test_sales) > 0:
                    x_test = np.arange(len(train_sales), len(sales_ts))
                    test_forecast = np.polyval(coeffs, x_test).tolist()
                else:
                    test_forecast = []

                return self.compile_forecast_results(sku, model_name, sales_ts, train_sales,
                                                   test_sales, train_forecast, test_forecast,
                                                   forecast_values, forecast_horizon)

            elif model_name == 'Naive':
                last_value = train_sales.iloc[-1]
                forecast_values = [last_value] * forecast_horizon
                train_forecast = train_sales.shift(1)
                test_forecast = [last_value] * len(test_sales) if len(test_sales) > 0 else []

                return self.compile_forecast_results(sku, model_name, sales_ts, train_sales,
                                                   test_sales, train_forecast, test_forecast,
                                                   forecast_values, forecast_horizon)

            # Generate forecasts for statsmodels/prophet models
            if model_name == 'Prophet':
                future_periods = len(train_sales) + forecast_horizon
                future_dates = [start_date + pd.DateOffset(months=i) for i in range(future_periods)]
                future_df = pd.DataFrame({'ds': future_dates})

                forecast_data = fitted_model.predict(future_df)
                forecast_values = forecast_data['yhat'][-forecast_horizon:].tolist()
                train_forecast = forecast_data['yhat'][:len(train_sales)]

                if len(test_sales) > 0:
                    test_end_idx = len(train_sales) + len(test_sales)
                    test_forecast = forecast_data['yhat'][len(train_sales):test_end_idx].tolist()
                else:
                    test_forecast = []
            else:
                # Statsmodels forecasts
                forecast_values = fitted_model.forecast(steps=forecast_horizon).tolist()
                train_forecast = fitted_model.fittedvalues
                test_forecast = fitted_model.forecast(steps=len(test_sales)).tolist() if len(test_sales) > 0 else []

            return self.compile_forecast_results(sku, model_name, sales_ts, train_sales,
                                               test_sales, train_forecast, test_forecast,
                                               forecast_values, forecast_horizon)

        except Exception as e:
            print(f"❌ Error in forecasting: {str(e)}")
            return None

    def compile_forecast_results(self, sku, model_name, sales_ts, train_sales,
                                test_sales, train_forecast, test_forecast, forecast_values, forecast_horizon):
        """Compile comprehensive forecast results"""

        # Calculate forecast accuracy metrics if test data available
        if len(test_forecast) > 0 and len(test_sales) > 0:
            min_len = min(len(test_forecast), len(test_sales))
            test_forecast_arr = np.array(test_forecast[:min_len])
            test_sales_arr = test_sales.values[:min_len]

            # Avoid division by zero
            mask = test_sales_arr > 0
            if np.any(mask):
                mape = np.mean(np.abs((test_sales_arr[mask] - test_forecast_arr[mask]) / test_sales_arr[mask])) * 100
            else:
                mape = np.nan

            rmse = np.sqrt(mean_squared_error(test_sales_arr, test_forecast_arr))
            mae = mean_absolute_error(test_sales_arr, test_forecast_arr)
        else:
            mape = rmse = mae = np.nan

        # Analyze forecast components
        forecast_ts = pd.Series(forecast_values)

        # Calculate forecast level, trend, seasonality
        forecast_level = forecast_ts.mean()
        forecast_trend_slope = np.polyfit(range(len(forecast_ts)), forecast_ts.values, 1)[0] if len(forecast_ts) > 1 else 0
        forecast_variability = forecast_ts.std()

        # Try to detect seasonality in forecast if enough periods
        forecast_seasonality = 'N/A'
        if len(forecast_values) >= 12:
            try:
                # Simple seasonality check - compare first half to second half patterns
                mid_point = len(forecast_values) // 2
                first_half_mean = np.mean(forecast_values[:mid_point])
                second_half_mean = np.mean(forecast_values[mid_point:])
                seasonal_variation = abs(first_half_mean - second_half_mean) / forecast_level if forecast_level > 0 else 0

                if seasonal_variation > 0.1:
                    forecast_seasonality = f"Detected (variation: {seasonal_variation:.3f})"
                else:
                    forecast_seasonality = f"Minimal (variation: {seasonal_variation:.3f})"
            except:
                forecast_seasonality = "Analysis unavailable"

        # Create future periods
        last_period = sales_ts.index[-1]
        future_periods = list(range(last_period + 1, last_period + 1 + forecast_horizon))

        return {
            'SKU': sku,
            'Model': model_name,
            'Formula_Used': self.get_model_formula(model_name),
            'Forecast_Horizon': forecast_horizon,
            'Historical_Sales': sales_ts,
            'Train_Data': train_sales,
            'Test_Data': test_sales,
            'Train_Forecast': train_forecast,
            'Test_Forecast': test_forecast,
            'Future_Forecast': forecast_values,
            'Future_Periods': future_periods,
            'Forecast_Level': round(forecast_level, 2),
            'Forecast_Trend': round(forecast_trend_slope, 4),
            'Forecast_Seasonality': forecast_seasonality,
            'Forecast_Variability': round(forecast_variability, 2),
            'MAPE': round(mape, 2) if not np.isnan(mape) else 'N/A',
            'RMSE': round(rmse, 2) if not np.isnan(rmse) else 'N/A',
            'MAE': round(mae, 2) if not np.isnan(mae) else 'N/A'
        }

    def get_model_formula(self, model_name):
        """Get the mathematical formula for the forecasting model"""
        formulas = {
            'Simple Exponential Smoothing': 'F(t+1) = α × X(t) + (1-α) × F(t)',
            'Double Exponential Smoothing': 'F(t+1) = L(t) + T(t), L(t) = α × X(t) + (1-α) × (L(t-1) + T(t-1)), T(t) = β × (L(t) - L(t-1)) + (1-β) × T(t-1)',
            'Triple Exponential Smoothing': 'F(t+1) = (L(t) + T(t)) × S(t+1-p), with level L(t), trend T(t), and seasonal S(t) components',
            'ARIMA': 'ARIMA(p,d,q): (1-φ₁B-...-φₚBᵖ)(1-B)ᵈXₜ = (1+θ₁B+...+θₑBᵠ)εₜ',
            'Seasonal ARIMA': 'SARIMA(p,d,q)(P,D,Q)s: Seasonal extension of ARIMA with period s',
            'Prophet': 'y(t) = g(t) + s(t) + h(t) + εₜ, where g(t) is trend, s(t) is seasonality, h(t) is holidays',
            'Moving Average': 'F(t+1) = (X(t) + X(t-1) + ... + X(t-n+1)) / n',
            'Linear Trend': 'F(t) = a + b × t, where a is intercept and b is slope',
            'Naive': 'F(t+1) = X(t)'
        }
        return formulas.get(model_name, 'Formula not available')

    def display_forecast_results(self, results):
        """Display comprehensive forecast results"""
        print(f"✅ {results['Model']} Forecast Results for {results['SKU']}")
        print("=" * 70)

        # Display model formula
        print(f"📐 Model Formula: {results['Formula_Used']}")
        print()

        # Display metrics
        if results['MAPE'] != 'N/A':
            print(f"📊 Model Validation Metrics:")
            print(f"   MAPE: {results['MAPE']}%")
            print(f"   RMSE: {results['RMSE']}")
            print(f"   MAE:  {results['MAE']}")

            # Accuracy interpretation
            if results['MAPE'] != 'N/A':
                if results['MAPE'] < 10:
                    accuracy = "Excellent"
                elif results['MAPE'] < 20:
                    accuracy = "Good"
                elif results['MAPE'] < 30:
                    accuracy = "Fair"
                else:
                    accuracy = "Poor"
                print(f"   Accuracy Rating: {accuracy}")

        # Forecast analysis
        print(f"\n🔮 Forecast Analysis:")
        print(f"   Forecasted Level: {results['Forecast_Level']}")
        print(f"   Trend Component: {results['Forecast_Trend']}")
        print(f"   Seasonality: {results['Forecast_Seasonality']}")
        print(f"   Forecast Variability: {results['Forecast_Variability']}")

        # Create visualizations
        fig, axes = plt.subplots(2, 1, figsize=(15, 12))

        # Main forecast plot
        ax1 = axes[0]

        # Plot historical data
        ax1.plot(results['Historical_Sales'].index, results['Historical_Sales'].values,
                label='Historical Sales', color='blue', linewidth=2, marker='o', markersize=4)

        # Plot forecasts
        if hasattr(results['Train_Forecast'], 'index'):
            ax1.plot(results['Train_Forecast'].index, results['Train_Forecast'].values,
                    label='Model Fit', color='orange', linewidth=2, alpha=0.8)
        elif hasattr(results['Train_Forecast'], '__len__'):
            train_periods = results['Train_Data'].index
            ax1.plot(train_periods, results['Train_Forecast'],
                    label='Model Fit', color='orange', linewidth=2, alpha=0.8)

        if len(results['Test_Forecast']) > 0 and len(results['Test_Data']) > 0:
            ax1.plot(results['Test_Data'].index, results['Test_Forecast'],
                    label='Validation Forecast', color='red', linewidth=2, marker='^', markersize=4)

        # Plot future forecast
        ax1.plot(results['Future_Periods'], results['Future_Forecast'],
                label='Future Forecast', color='purple', linewidth=3, marker='*', markersize=8)

        ax1.set_title(f'{results["SKU"]} - {results["Model"]} Forecast', fontweight='bold', fontsize=14)
        ax1.set_xlabel('Period')
        ax1.set_ylabel('Sales')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        # Residuals analysis
        if len(results['Test_Forecast']) > 0 and len(results['Test_Data']) > 0:
            ax2 = axes[1]
            min_len = min(len(results['Test_Forecast']), len(results['Test_Data']))
            residuals = results['Test_Data'].values[:min_len] - np.array(results['Test_Forecast'][:min_len])

            ax2.plot(results['Test_Data'].index[:min_len], residuals,
                    color='red', marker='o', linewidth=2, label='Residuals')
            ax2.axhline(y=0, color='black', linestyle='--', alpha=0.5)
            ax2.set_title('Forecast Residuals (Actual - Predicted)', fontweight='bold')
            ax2.set_xlabel('Period')
            ax2.set_ylabel('Residuals')
            ax2.grid(True, alpha=0.3)
        else:
            axes[1].text(0.5, 0.5, 'No validation data available for residual analysis',
                        ha='center', va='center', transform=axes[1].transAxes, fontsize=12)
            axes[1].set_xticks([])
            axes[1].set_yticks([])

        plt.tight_layout()
        plt.show()

        # Display future values table
        print("\n🔮 Future Forecast Values:")
        print("-" * 40)

        forecast_df = pd.DataFrame({
            'Period': results['Future_Periods'],
            'Forecasted_Sales': [round(val, 2) for val in results['Future_Forecast']]
        })
        print(forecast_df.to_string(index=False))

        # Store results
        self.forecast_results[f"{results['SKU']}_{results['Model']}"] = results

    def create_forecast_download_button(self, results):
        """Create download button for forecast results"""
        print("\n💾 DOWNLOAD FORECAST RESULTS")
        print("=" * 40)

        download_button = widgets.Button(
            description='📥 Download Forecast Report',
            button_style='warning',
            layout=widgets.Layout(width='250px', height='40px')
        )

        def download_forecast(b):
            try:
                output = io.BytesIO()

                with pd.ExcelWriter(output, engine='openpyxl') as writer:
                    # Forecast Summary
                    summary = {
                        'Metric': ['SKU', 'Forecasting Model', 'Formula Used', 'Forecast Horizon (Periods)',
                                  'Forecasted Level', 'Trend Component', 'Seasonality', 'Forecast Variability',
                                  'Validation MAPE (%)', 'Validation RMSE', 'Validation MAE'],
                        'Value': [
                            results['SKU'],
                            results['Model'],
                            results['Formula_Used'],
                            results['Forecast_Horizon'],
                            results['Forecast_Level'],
                            results['Forecast_Trend'],
                            results['Forecast_Seasonality'],
                            results['Forecast_Variability'],
                            results['MAPE'],
                            results['RMSE'],
                            results['MAE']
                        ]
                    }

                    summary_df = pd.DataFrame(summary)
                    summary_df.to_excel(writer, sheet_name='Forecast_Summary', index=False)

                    # Combine Historical, Validation, and Future Forecast data
                    combined_data = pd.DataFrame()

                    # Add Historical Data
                    historical_df = pd.DataFrame({
                        'Period': results['Historical_Sales'].index,
                        'Actual_Sales': results['Historical_Sales'].values,
                        'Model_Fit': None, # Placeholder, will be filled below
                        'Predicted_Sales': None, # Placeholder
                        'Residual': None, # Placeholder
                        'Absolute_Error': None, # Placeholder
                        'Percentage_Error': None # Placeholder
                    })

                    # Add Model Fit to historical data
                    if hasattr(results['Train_Forecast'], 'values'):
                        train_fit = list(results['Train_Forecast'].values) + [None] * (len(historical_df) - len(results['Train_Forecast']))
                        historical_df['Model_Fit'] = train_fit[:len(historical_df)]
                    elif hasattr(results['Train_Forecast'], '__len__'):
                        train_fit = list(results['Train_Forecast']) + [None] * (len(historical_df) - len(results['Train_Forecast']))
                        historical_df['Model_Fit'] = train_fit[:len(historical_df)]


                    combined_data = pd.concat([combined_data, historical_df])

                    # Add Validation Results (if available)
                    if len(results['Test_Data']) > 0 and len(results['Test_Forecast']) > 0:
                        min_len = min(len(results['Test_Data']), len(results['Test_Forecast']))
                        validation_df = pd.DataFrame({
                            'Period': results['Test_Data'].index[:min_len],
                            'Actual_Sales': results['Test_Data'].values[:min_len],
                            'Model_Fit': None, # Placeholder
                            'Predicted_Sales': results['Test_Forecast'][:min_len],
                            'Residual': results['Test_Data'].values[:min_len] - np.array(results['Test_Forecast'][:min_len]),
                            'Absolute_Error': np.abs(results['Test_Data'].values[:min_len] - np.array(results['Test_Forecast'][:min_len])),
                            'Percentage_Error': np.abs((results['Test_Data'].values[:min_len] - np.array(results['Test_Forecast'][:min_len])) / results['Test_Data'].values[:min_len]) * 100
                        })
                        combined_data = pd.concat([combined_data, validation_df])

                    # Add Future Forecast Values
                    forecast_df = pd.DataFrame({
                        'Period': results['Future_Periods'],
                        'Actual_Sales': None, # Placeholder
                        'Model_Fit': None, # Placeholder
                        'Predicted_Sales': results['Future_Forecast'],
                        'Residual': None, # Placeholder
                        'Absolute_Error': None, # Placeholder
                        'Percentage_Error': None # Placeholder
                    })
                    combined_data = pd.concat([combined_data, forecast_df])

                    # Sort by Period to ensure correct order
                    combined_data = combined_data.sort_values(by='Period').reset_index(drop=True)

                    # Add metadata columns
                    combined_data['SKU'] = results['SKU']
                    combined_data['Model'] = results['Model']
                    combined_data['Formula_Used'] = results['Formula_Used']
                    combined_data['Level'] = results['Forecast_Level']
                    combined_data['Trend_Component'] = results['Forecast_Trend']
                    combined_data['Seasonality'] = results['Forecast_Seasonality']
                    combined_data['Variability'] = results['Forecast_Variability']


                    combined_data.to_excel(writer, sheet_name='Full_Forecast_Data', index=False)


                output.seek(0)

                # Create download link
                b64 = base64.b64encode(output.getvalue()).decode()
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                filename = f"full_forecast_data_{results['SKU']}_{results['Model']}_{timestamp}.xlsx"

                download_link = f'<a download="{filename}" href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64}">Click here to download full forecast report</a>'
                display(HTML(download_link))

                print("✅ Full forecast report ready for download!")

            except Exception as e:
                print(f"❌ Error creating forecast download: {str(e)}")

        download_button.on_click(download_forecast)
        display(download_button)

# Initialize and start the tool
tool = DemandSalesAnalysisTool()

print("🚀 DEMAND vs SALES ANALYSIS & FORECASTING TOOL")
print("=" * 70)
print("This tool will guide you through:")
print("1️⃣ Upload demand and sales data (or download sample file)")
print("2️⃣ Analyze patterns and calculate metrics (Level, Trend, Seasonality, MAPE, RMSE)")
print("3️⃣ Suggest forecasting models based on sales patterns")
print("4️⃣ Generate forecasts with formulas and download complete results")
print()

# Start the workflow
tool.start_analysis()